# 17b_score_fingpt.ipynb

**Purpose:** Score the 796,038 scoring-input headlines with FinGPT (adapter `FinGPT/fingpt-mt_llama2-7b_lora` on base `NousResearch/Llama-2-7b-hf`, 4-bit nf4) on a Colab GPU, producing `p_pos`, `p_neu`, `p_neg` from a softmax over the three answer-token logits.

**Inputs:** `/content/drive/MyDrive/thesis_finbert/finbert_input.csv` — the same file 16b reads, uploaded from `Data/colab/finbert_input.csv`, which `pipeline/16a_export_for_colab.py` produces; SHA256 and row count are asserted on load.

**Outputs:** `/content/drive/MyDrive/thesis_fingpt/fingpt_scores_partial.csv` (per-chunk checkpoint) and `/content/drive/MyDrive/thesis_fingpt/fingpt_scores.csv` (final, ordered by `hid`). The final file is copied back to `Data/colab/fingpt_scores.csv` and consumed by `pipeline/17c_fingpt_aggregate_freeze.py`.

**Notes**
- Runtime: any CUDA GPU; after three chunks a `RATE CHECK` line estimates the full-run hours (a rented GPU can take over, since the partial file lives on Drive).
- Resume: the full run appends to the partial file every 10,000 rows; re-running the notebook skips the `hid`s already scored.
- Scoring: FinGPT prompt template with a trailing space after `Answer: `, bare-word class token ids, logits-only softmax over the three class tokens (no generation); the smoke test, negative canary and padding-equivalence check must pass before the full run.
- Batch sizing: starts at 32 and halves automatically on OOM (floor 1); left padding keeps the answer position last for every row.
- `hid` is the row position assigned by 16a and the join key used by 17c.

In [ ]:
# Step 1: install dependencies (pandas<2.3 pinned to avoid a Colab clash), check versions and GPU
%pip install -q "pandas<2.3" transformers peft accelerate bitsandbytes
import hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch, transformers, peft, bitsandbytes
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| peft', peft.__version__, '| bitsandbytes', bitsandbytes.__version__,
      '| pandas', pd.__version__)
assert pd.__version__ < '2.3', 'pandas pin failed - restart runtime and re-run this cell'
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2: mount Drive and verify the 16b input file (SHA256, row count, hid range)
from google.colab import drive
drive.mount('/content/drive')

INPUT = Path('/content/drive/MyDrive/thesis_finbert/finbert_input.csv')  # S4b upload, REUSED
OUT_DIR = Path('/content/drive/MyDrive/thesis_fingpt')
OUT_DIR.mkdir(parents=True, exist_ok=True)
PARTIAL, FINAL = OUT_DIR / 'fingpt_scores_partial.csv', OUT_DIR / 'fingpt_scores.csv'

INPUT_SHA = 'e87ba39d4f43081736e03d9fa4a9e89df54658e3d08dea3e48d51ed01ebef346'
N_EXPECTED = 796_038

sha = hashlib.sha256(INPUT.read_bytes()).hexdigest()
assert sha == INPUT_SHA, f'input SHA mismatch: {sha}'
inp = pd.read_csv(INPUT, dtype={'hid': 'int64', 'PERMNO': 'int64', 'target_week': str, 'headline': str})
assert len(inp) == N_EXPECTED and inp['hid'].is_unique
assert inp['hid'].min() == 0 and inp['hid'].max() == N_EXPECTED - 1
print(f'input OK: {len(inp):,} rows, SHA verified {sha[:8]}...')

In [ ]:
# Step 3: load the Llama-2-7B base in 4-bit nf4, attach the FinGPT LoRA adapter, set left padding
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_ID = 'NousResearch/Llama-2-7b-hf'
ADAPTER_ID = 'FinGPT/fingpt-mt_llama2-7b_lora'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE_ID)
model = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map='auto')
model = PeftModel.from_pretrained(model, ADAPTER_ID)
model.eval()
tok.pad_token = tok.eos_token
tok.padding_side = 'left'      # decoder model: answer position = LAST for every row
assert tok.padding_side == 'left'
print('model + adapter loaded (4-bit nf4), left-padding set')

In [ ]:
# Step 4: prompt template, bare-word class token ids and the logits-only scorer
# Template: 'Instruction: What is the sentiment of this news? Please choose an answer
# from {negative/neutral/positive}.\nInput: <headline>\nAnswer: ' (trailing space kept,
# so the class words are tokenised as bare words).
PROMPT_PREFIX = ('Instruction: What is the sentiment of this news? '
                 'Please choose an answer from {negative/neutral/positive}.\n'
                 'Input: ')
PROMPT_SUFFIX = '\nAnswer: '
assert PROMPT_SUFFIX.endswith(' ') and '{negative/neutral/positive}' in PROMPT_PREFIX

def build_prompt(headline):
    return PROMPT_PREFIX + headline + PROMPT_SUFFIX

def first_id(word):
    ids = tok(word, add_special_tokens=False)['input_ids']
    # each bare word must map to a single token id; otherwise stop (tokenizer drift)
    assert len(ids) == 1, f'STOP: {word!r} tokenizes to {ids} (not a single id)'
    return ids[0]

ID_POS, ID_NEU, ID_NEG = first_id('positive'), first_id('neutral'), first_id('negative')
assert len({ID_POS, ID_NEU, ID_NEG}) == 3
print(f'bare-word token ids: positive={ID_POS} neutral={ID_NEU} negative={ID_NEG}')

@torch.no_grad()
def score_batch(headlines):
    """Logits-only scoring: single forward pass, last-position logits,
    softmax over exactly the three answer-token logits. No generation."""
    prompts = [build_prompt(h) for h in headlines]
    enc = tok(prompts, return_tensors='pt', padding=True).to(model.device)
    logits = model(**enc).logits[:, -1, :].float()
    three = logits[:, [ID_POS, ID_NEU, ID_NEG]]
    return torch.softmax(three, dim=-1).cpu().numpy()   # cols: p_pos, p_neu, p_neg

In [ ]:
# Step 5: smoke test plus a negative-strength canary; both must pass before the full run
smoke = ['Company files for bankruptcy protection',
         'Company beats earnings estimates, raises guidance',
         'Company announces quarterly dividend',
         inp['headline'].iloc[0],
         inp['headline'].iloc[500000]]
p = score_batch(smoke)
for t, (a, b, c) in zip(smoke, p):
    print(f'p_pos {a:.3f}  p_neu {b:.3f}  p_neg {c:.3f}  | {t[:90]}')
assert p[0, 2] > max(p[0, 0], p[0, 1]), 'SMOKE FAIL: bankruptcy not negative-dominant'
assert p[0, 2] > 0.6, ('CANARY FAIL: p_neg <= 0.6 - the ~0.4-0.5 mush signature of '
                       'trailing-space/template drift. STOP and report; do not adapt.')
assert p[1, 0] > max(p[1, 1], p[1, 2]), 'SMOKE FAIL: beats-guidance not positive-dominant'
print('SMOKE + CANARY PASSED')

In [ ]:
# Step 6: left-padding equivalence check, batched vs single-row scores on mixed-length rows
lens = inp['headline'].str.len()
test_hids = [int(lens.idxmin()), int(lens.idxmax()), 250000]   # short, long, mid
texts = inp.loc[test_hids, 'headline'].tolist()
p_batch = score_batch(texts)
p_single = np.vstack([score_batch([t]) for t in texts])
diff = np.abs(p_batch - p_single).max()
print(f'batched vs single-row max |diff| over 3 mixed-length rows: {diff:.2e}')
# tolerance 5e-3 absorbs cross-GPU numeric noise in 4-bit inference; a positional bug
# (broken left padding / wrong answer position) gives |diff| >= 0.05
assert diff < 5e-3, 'PADDING FAIL: batched != single - left-padding broken'
print('PADDING EQUIVALENCE PASSED (answer position = last for every row)')

In [ ]:
# Step 7: full run in 10k-row chunks, Drive checkpoint per chunk, resume by hid, OOM backoff
CHUNK = 10_000
BATCH = 32                      # auto-halves on OOM, floor 1

def score_with_backoff(texts):
    global BATCH
    out = np.empty((len(texts), 3), dtype=np.float32)
    i = 0
    while i < len(texts):
        bs = min(BATCH, len(texts) - i)
        try:
            out[i:i + bs] = score_batch(texts[i:i + bs])
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            assert BATCH > 1, 'OOM at batch size 1 - cannot proceed'
            BATCH = BATCH // 2
            print(f'  OOM -> batch size halved to {BATCH}')
    return out

done = set()
if PARTIAL.exists():
    done = set(pd.read_csv(PARTIAL, usecols=['hid'])['hid'].tolist())
    print(f'resuming: {len(done):,} hids already scored')
todo = inp[~inp['hid'].isin(done)].reset_index(drop=True)
print(f'to score: {len(todo):,}')

t0, session_chunks = time.time(), 0
for c0 in range(0, len(todo), CHUNK):
    sub = todo.iloc[c0:c0 + CHUNK]
    out = score_with_backoff(sub['headline'].tolist())
    pd.DataFrame({'hid': sub['hid'].to_numpy(), 'p_pos': out[:, 0],
                  'p_neu': out[:, 1], 'p_neg': out[:, 2]}).to_csv(
        PARTIAL, mode='a', header=not PARTIAL.exists(), index=False)
    session_chunks += 1
    n_done = len(done) + c0 + len(sub)
    rate = (c0 + len(sub)) / max(time.time() - t0, 1e-9)
    eta_h = (len(todo) - c0 - len(sub)) / max(rate, 1e-9) / 3600
    print(f'checkpoint: {n_done:,}/{N_EXPECTED:,} | {rate:,.1f} rows/s | '
          f'ETA {eta_h:.2f} h | batch {BATCH}')
    if session_chunks == 3:
        print(f'\nRATE CHECK: {rate:,.1f} rows/s -> full run ~ '
              f'{N_EXPECTED / rate / 3600:.1f} hours on this GPU. Decide: '
              f'continue here, or stop and resume on a rented GPU '
              f'(resume is free - the partial file is on Drive).\n')
print('FULL RUN DONE')

In [ ]:
# Step 8: consolidate the partial file into fingpt_scores.csv with the same checks as 16b
part = pd.read_csv(PARTIAL)
n_dup = int(part.duplicated('hid').sum())
if n_dup:
    print(f'note: {n_dup} duplicate hids (restart overlap) - keeping first')
    part = part.drop_duplicates('hid', keep='first')
part = part.sort_values('hid').reset_index(drop=True)
assert len(part) == N_EXPECTED, f'{len(part):,} rows - chunks missing, re-run previous cell'
assert part['hid'].is_unique and part['hid'].min() == 0 and part['hid'].max() == N_EXPECTED - 1
P = part[['p_pos', 'p_neu', 'p_neg']]
assert P.isna().sum().sum() == 0
assert ((P >= 0) & (P <= 1)).all().all()
assert ((P.sum(axis=1) - 1).abs() < 1e-4).all()
part.to_csv(FINAL, index=False)
print(f'wrote {FINAL}: {len(part):,} rows (ordered by hid)')
print('per-column means:', P.mean().round(4).to_dict())
print(part.head(3).to_string(index=False))
print('\nDOWNLOAD fingpt_scores.csv -> <repo>/Data/colab/ -> run Code/17c_fingpt_aggregate_freeze.py')